In [0]:
from pyspark.sql.functions import col, when, rand
from pyspark.sql.functions import create_map, lit
from itertools import chain
import mlflow
import mlflow.sklearn
from datetime import datetime

# ── Step 1: Read gold table ──────────────────────────────────
df = spark.table("workspace.default.gold_diabetic")
print("Total patients:", df.count())

# ── Step 2: Select feature columns ──────────────────────────
feature_cols = [
    "age_numeric", "time_in_hospital", "num_lab_procedures",
    "num_procedures", "num_medications", "number_diagnoses",
    "number_outpatient", "number_emergency", "number_inpatient",
    "total_prior_visits", "high_utilizer", "num_active_meds",
    "on_insulin", "med_changed"
]

# Convert to pandas for scoring with our sklearn model
pandas_df = df.select(feature_cols + ["patient_nbr", "age", "race", 
                                       "gender", "readmitted"]).toPandas()

print("Features ready:", pandas_df.shape)

In [0]:
import pickle
import pandas as pd
import numpy as np

# ── Step 3: Load the saved model ────────────────────────────
# We need to upload our model file to Databricks first
# For now we recreate a simple model using the gold table data

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Prepare features
X = pandas_df[feature_cols].fillna(0)
y = pandas_df["readmitted"].apply(lambda x: 0 if x == "NO" else 1)

# Train model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, 
    learning_rate=0.1, random_state=42
)
model.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print(f"Model AUC: {auc:.4f}")

# ── Step 4: Score all patients ───────────────────────────────
pandas_df["risk_score"] = (model.predict_proba(X)[:,1] * 100).round(1)
pandas_df["risk_tier"] = pd.cut(pandas_df["risk_score"],
    bins=[0, 30, 60, 80, 100],
    labels=["Low", "Medium", "High", "Critical"])

print("\nRisk Tier Distribution:")
print(pandas_df["risk_tier"].value_counts())

In [0]:
# ── Step 5: Fix risk tier and save ───────────────────────────
# Convert categorical to string first
pandas_df["risk_score"] = pandas_df["risk_score"].astype(float)
pandas_df["risk_tier"] = pandas_df["risk_tier"].astype(str)

# Convert to Spark DataFrame
scored_spark = spark.createDataFrame(pandas_df[[
    "patient_nbr", "age", "race", "gender",
    "risk_score", "risk_tier", "readmitted"
]])

# Save as Delta table
scored_spark.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.daily_risk_scores")

print("✅ Risk scores saved!")
print("Total patients scored:", scored_spark.count())

# ── Step 6: Save high risk alerts ───────────────────────────
alerts = pandas_df[pandas_df["risk_tier"].isin(["Critical", "High"])] \
    .sort_values("risk_score", ascending=False)

alerts_spark = spark.createDataFrame(alerts[[
    "patient_nbr", "age", "race", "gender",
    "risk_score", "risk_tier", "num_lab_procedures",
    "num_medications", "on_insulin", "total_prior_visits"
]])

alerts_spark.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.high_risk_alerts")

print("✅ High risk alerts saved!")
print("High risk patients:", alerts_spark.count())
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")